In [37]:
import os
import sys
import yaml

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

from sc_exp_design.models import TargetPredictionModel

In [38]:
BASE_DIR = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC"
sys.path.insert(0, os.path.join(BASE_DIR, "scripts"))
from data_utils import get_protocol_tranformations

In [39]:
RESULTS_DIR = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/results"
TARGET_PREDICTION_MODEL_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_21-22-45/dashing-frog-1258_TargetPredictionModel.pkl"
ANNOTATION_CONFIG_PATH = os.path.join(
    BASE_DIR,
    "generative_modeling/conditional_flow_matching/config/annotation/default.yaml"
)

In [40]:
with open(ANNOTATION_CONFIG_PATH, "r") as fb:
    annotation_config = yaml.safe_load(fb)
print(annotation_config.keys())

dict_keys(['protocol_columns', 'protocol_obs_key_added', 'protocol_sep', 'one_hot_uns_key_added', 'protocol_obsm_key', 'scatter_columns', 'base_sample_rep', 'scatter_obsm_key', 'concat_obsm_key', 'log1p_exp_cols', 'log21p_exp_cols'])


In [41]:
column2tranform = get_protocol_tranformations(
    annotation_config["protocol_columns"],
    log1p_exp_cols=annotation_config["log1p_exp_cols"],
    log21p_exp_cols=annotation_config["log21p_exp_cols"],
    inverse=True
)
len(column2tranform)

15

In [42]:
target_prediction_model = TargetPredictionModel.load(
    TARGET_PREDICTION_MODEL_PATH
)
# Prepare label encoder
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = ct_le.classes_.tolist()

In [43]:
os.listdir(RESULTS_DIR)

['Early_Pro-B',
 'candidates_with_eval.csv',
 'CD14-Mono',
 'GMP-Cycle',
 'GMP-Neutro_CD16-Mono',
 'MgkPro',
 'EosBasoMastPre_2',
 'CyclingProgenitor*',
 'MEP',
 'MLP',
 'EryPro',
 'pDCs_cDCs',
 'top_15_candidates.csv',
 'HSCs',
 'GMP-Neutro',
 'MPP',
 'EosBasoMastPre',
 'all_candidates.csv',
 'CD49c-Myeloid',
 'Pro-B',
 'EarlyGMP',
 'DCs-CD14']

In [44]:
def flatten_conf(conf_dict):
    return_dict = {}
    for key, value in conf_dict.items():
        if isinstance(value, dict):
            update_dict = {f"{key}:{k}": v for k, v in flatten_conf(value).items()}
        else:
            update_dict = {key: value}
        return_dict.update(update_dict)
    return return_dict


In [45]:
ct_results_dict = {}
dfs_all_candidates = []
ct_list = os.listdir(RESULTS_DIR)
for ct in ct_list:
    ct_dir = os.path.join(RESULTS_DIR, ct)
    if not os.path.isdir(ct_dir):
        continue
    ct_runs = os.listdir(ct_dir)
    ct_dfs_list = []
    print(f"Starting to process runs for {ct}, {len(ct_runs)} runs found")
    pbar = tqdm(range(len(ct_runs)))
    for run in ct_runs:
        run_dir = os.path.join(ct_dir, run)
        if not os.path.isdir(run_dir):
            continue
        pbar.set_description(f"Reading results for {ct}, {run}")
        pbar.update()

        # configuration file
        with open(os.path.join(run_dir, "config.yaml"), "r") as fb:
            conf_dict = yaml.safe_load(fb)
        fconf_dict = flatten_conf(conf_dict)

        inverse_results = np.load(
            os.path.join(run_dir, "inverse_results.npz")
        )
        loss_history = inverse_results["loss_history"]
        trajectory = inverse_results["trajectory"]

        fwd_results = np.load(
            os.path.join(run_dir, "fwd_results.npz")
        )
        ct_props = fwd_results["ct_probs"].mean(1)


        samples = np.maximum(trajectory[:, -1, :], 0)
        data_dict_transformed = {
            mol: samples[:, idx] for idx, mol in enumerate(annotation_config["protocol_columns"]) 
        }
        data_dict = {}
        for mol, val in data_dict_transformed.items():
            trnsf = column2tranform[mol]
            if trnsf is not None:
                val = trnsf(val)
            data_dict[mol] = val
        
        data_dict["loss"] = loss_history[:, -1]
        
        samples_df = pd.DataFrame(
            {
                **data_dict,
                **{f"{k}:rescaled":v for k, v in data_dict_transformed.items()},
                "loss": loss_history[:, -1],
                **{
                    f"{ct}_prop": ct_props[:, idx] for idx, ct in enumerate(classes)
                },
                **fconf_dict
            },
        )
        samples_df.index = samples_df.index.map(lambda e: f"{ct}:{run}:{e}")

        samples_df.to_csv(
            os.path.join(run_dir, "candidates.csv")
        )
        ct_dfs_list.append(samples_df)

    print(f"Creating concatenated results...")
    ct_df = pd.concat(
        ct_dfs_list, axis=0
    )
    print(f"Concatenated results: {ct_df.shape}")
    ct_df.to_csv(
        os.path.join(ct_dir, "all_candidates.csv")
    )
    dfs_all_candidates.append(ct_df)

dfs_all_candidates = pd.concat(
    dfs_all_candidates, axis=0
)
dfs_all_candidates.to_csv(
    os.path.join(RESULTS_DIR, "all_candidates.csv")
)

Starting to process runs for Early_Pro-B, 73 runs found


Reading results for EryPro, 2026-01-21_14-53-49_e3f8b2ed:  99%|█████████▊| 66/67 [00:28<00:00,  2.35it/s]

Creating concatenated results...
Concatenated results: (9000, 76)


/tmp/ipykernel_3853508/1026787483.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ct_df = pd.concat(


Starting to process runs for CD14-Mono, 73 runs found


...

/tmp/ipykernel_3853508/1026787483.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ct_df = pd.concat(


Starting to process runs for DCs-CD14, 73 runs found


Reading results for EarlyGMP, 2025-12-20_19-45-15_93230e69:  99%|█████████▊| 72/73 [00:21<00:00,  3.29it/s]


Creating concatenated results...
Concatenated results: (9000, 76)


/tmp/ipykernel_3853508/1026787483.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ct_df = pd.concat(
